In [6]:
import os
print("Current working directory:", os.getcwd())
print("Files here:", os.listdir())


Current working directory: C:\Users\hp\Desktop\Fynd_AI_Intern_Assessment
Files here: ['.ipynb_checkpoints', 'Task1_Rating_Prediction.ipynb', 'Untitled.ipynb', 'Untitled1.ipynb', 'Untitled3.ipynb', 'yelp.csv']


In [8]:
%%writefile prompts.py
PROMPT_ZERO_SHOT = """
You are a helpful assistant. Classify the following Yelp review into a star rating (1 to 5).
Return ONLY valid JSON with keys "predicted_stars" and "explanation".

Review:
<<REVIEW>>

JSON:
{
  "predicted_stars": ,
  "explanation": ""
}
"""

PROMPT_FEW_SHOT = """
You are an assistant that maps Yelp reviews to star ratings.

Example 1:
Review: "Food was terrible and cold, service rude."
Rating: 1

Example 2:
Review: "Great food, prompt staff, fair price."
Rating: 4

Example 3:
Review: "Decent spot, nothing special."
Rating: 3

Now classify the review below.

Review:
<<REVIEW>>

JSON:
{
  "predicted_stars": ,
  "explanation": ""
}
"""

PROMPT_STEPWISE = """
Step1: Output STAR: <1-5>
Step2: Output EXPLAIN: <short reasoning>
Step3: Output JSON:
{
 "predicted_stars": <number>,
 "explanation": "<text>"
}

Review:
<<REVIEW>>
"""



Writing prompts.py


In [12]:


import os
import json
import time
import pandas as pd
from tqdm.auto import tqdm
from typing import List, Dict
from jsonschema import validate, ValidationError
from prompts import PROMPT_ZERO_SHOT, PROMPT_FEW_SHOT, PROMPT_STEPWISE

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
DATA_PATH = "yelp.csv"

# Schema for output JSON (simple)
OUTPUT_SCHEMA = {
    "type": "object",
    "properties": {
        "predicted_stars": {"type": "integer", "minimum": 1, "maximum": 5},
        "explanation": {"type": "string"}
    },
    "required": ["predicted_stars", "explanation"]
}

def llm_call_openai(prompt: str, model="gpt-4o-mini", temperature=0.0, max_tokens=150):
    import openai
    openai.api_key = OPENAI_API_KEY
    resp = openai.ChatCompletion.create(
        model=model,
        messages=[{"role":"user","content":prompt}],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return resp["choices"][0]["message"]["content"]


PROMPT_ZERO_SHOT = """
You are a helpful assistant. Classify the following Yelp review into a star rating (1 to 5).
Return ONLY valid JSON with keys "predicted_stars" (integer 1-5) and "explanation" (brief reasoning).

Review:
<<REVIEW>>

JSON:
{
  "predicted_stars": ,
  "explanation": ""
}
"""

PROMPT_FEW_SHOT = """
You are an assistant that maps Yelp reviews to star ratings. Below are examples.

Example 1:
Review: "Food was terrible and cold, service rude."
Rating: 1
Explanation: Poor experience.

Example 2:
Review: "Great food, prompt staff, fair price."
Rating: 4
Explanation: Mostly positive.

Example 3:
Review: "Decent spot, nothing special."
Rating: 3
Explanation: Neutral sentiment.

Now classify the review below. Return ONLY JSON.

Review:
<<REVIEW>>

JSON:
{
  "predicted_stars": ,
  "explanation": ""
}
"""

PROMPT_STEPWISE = """
Step1: Output "STAR: <1-5>"
Step2: Output "EXPLAIN: <short explanation>"
Step3: Output JSON:
{
 "predicted_stars": <number>,
 "explanation": "<text>"
}

Review:
<<REVIEW>>
"""

with open("prompt_versions.json", "w") as f:
    json.dump({
        "zero_shot": PROMPT_ZERO_SHOT,
        "few_shot": PROMPT_FEW_SHOT,
        "stepwise": PROMPT_STEPWISE
    }, f, indent=2)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"{DATA_PATH} not found. Download the Kaggle dataset and save as {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
# Keep columns: 'text' (review) and 'stars' (true)
print("Dataset shape:", df.shape)
df_sample = df.sample(n=200, random_state=42).reset_index(drop=True)  # recommended ~200 rows

def run_prompt_experiment(prompt_template: str, samples: pd.DataFrame, llm_func=llm_call_openai):
    results = []
    for i, row in tqdm(samples.iterrows(), total=len(samples)):
        review_text = row['text'] if 'text' in row else row['review']
        true_star = int(row['stars'])
        prompt = prompt_template.replace("<<REVIEW>>", review_text)
        # call model
        try:
            output = llm_func(prompt)
        except Exception as e:
            output = f"LLM_ERROR: {e}"
        # Try to parse JSON from output
        pred_obj = None
        json_valid = False
        parsed_error = None
        try:
            parsed = json.loads(output)
            validate(instance=parsed, schema=OUTPUT_SCHEMA)
            pred_obj = parsed
            json_valid = True
        except (json.JSONDecodeError, ValidationError) as e:
            parsed_error = str(e)
            # try to salvage by extracting first JSON-looking block
            import re
            m = re.search(r"\{.*\}", output, re.DOTALL)
            if m:
                try:
                    parsed = json.loads(m.group(0))
                    validate(instance=parsed, schema=OUTPUT_SCHEMA)
                    pred_obj = parsed
                    json_valid = True
                except Exception as e2:
                    parsed_error = parsed_error + " | salvage_failed: " + str(e2)

        predicted = pred_obj["predicted_stars"] if pred_obj else None
        explanation = pred_obj["explanation"] if pred_obj else (output[:200] if output else "")
        results.append({
            "index": i,
            "true_star": true_star,
            "predicted_star": predicted,
            "json_valid": json_valid,
            "explanation_raw": explanation,
            "raw_output": output,
            "parse_error": parsed_error
        })
    return pd.DataFrame(results)


Dataset shape: (10000, 10)


In [14]:
res_zero = run_prompt_experiment(PROMPT_ZERO_SHOT, df_sample)
res_few = run_prompt_experiment(PROMPT_FEW_SHOT, df_sample)
res_step = run_prompt_experiment(PROMPT_STEPWISE, df_sample)

res_zero.to_csv("results_zero_shot.csv", index=False)
res_few.to_csv("results_few_shot.csv", index=False)
res_step.to_csv("results_stepwise.csv", index=False)


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

In [15]:
print("Zero shot:", summarize_results(res_zero))
print("Few shot:", summarize_results(res_few))
print("Stepwise:", summarize_results(res_step))


Zero shot: {'total': 200, 'json_valid_rate': np.float64(0.0), 'accuracy_on_valid': 0.0}
Few shot: {'total': 200, 'json_valid_rate': np.float64(0.0), 'accuracy_on_valid': 0.0}
Stepwise: {'total': 200, 'json_valid_rate': np.float64(0.0), 'accuracy_on_valid': 0.0}
